<a href="https://colab.research.google.com/github/joseportocarrero-stack/DataScience-Homework/blob/main/FGD_C28R_1A_LABD6_Portocarrero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio Dirigido N.° 06
## Semana 6: Limpieza de Datos — Pipeline sobre `ventas_original`

| Campo | Detalle |
|---|---|
| **Curso** | Fundamentos de Gestión de Datos |
| **Docente** | Pilar Rocío Sayán Mejía |
| **Semana** | 6 |
| **Caso** | Abarrotes Esperanza |
| **Tabla de trabajo** | `ventas_original` |
| **Entregable** | Notebook ejecutado con respuestas completas |

> **Instrucción:** Este notebook trabaja directamente desde `ventas_original`. No asumas que existen tablas normalizadas. Ejecuta todas las celdas en orden, completa cada respuesta y exporta el dataset limpio al finalizar.

---
## Revisión de conocimientos de la semana

Antes de ejecutar el código, completa el siguiente cuadro con tus propias palabras.

| Concepto / Principio | Definición — responde con tus propias palabras |
|---|---|
| **1. Pipeline de limpieza de datos** |Canal de flujo de datos automatizado para aplicar cambios de formato, eliminación de duplicados y relleno de valores faltantes. |
| **2. Valores faltantes e imputación** |Un valor faltante es una celda en blanco, y la imputación es el proceso de cambiar el valor de celdas con contenidos nulos, inconsistentes y outliers. |
| **3. Outliers y método IQR** |Los outliers son valores extremos o atípicos, muy por encima o por debajo de una distribución normal de datos. El rango intercuartil es un método de agrupación de datos, consistente en restar Q3 - Q1, de tal manera que el resultado agrupa al 50% central de todos los datos analizados.|
| **4. Normalización Min-Max** |Conversión de los valores de un atributo, casi siempre a una escala de 0 a 1, para comparar valores con escalas muy diferentes.|
| **5. Tabla desnormalizada** |Puede referirse a una tabla que usualmente contiene toda la información que maneja una empresa o proyecto, pudiendo presentar inconsistencias o redundancia de registros. También puede referirse a una tabla que, habiendo pasado por los procesos de normalización, vuelve a juntarse para aumentar la rapidez de las consultas. |


---
## Actividad 1. Carga de librerías y base de datos desde GitHub

In [ ]:
try:
    import requests
except ModuleNotFoundError:
    requests = None

import sqlite3
import pandas as pd
import numpy as np
import warnings
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError:
    plt = None
    sns = None

url = "https://raw.githubusercontent.com/Rociosayan/PMD1_Fundamentos_Gestion_Datos/main/casos/16_abarrotes_ventas_inventario/abarrotes_ventas_inventario.db"

if requests is not None:
    r = requests.get(url)
    r.raise_for_status()
    contenido = r.content
else:
    from urllib.request import urlopen
    with urlopen(url) as respuesta:
        contenido = respuesta.read()

open("abarrotes_ventas_inventario.db", "wb").write(contenido)

conn = sqlite3.connect("abarrotes_ventas_inventario.db")

df = pd.read_sql_query("SELECT * FROM ventas_original LIMIT 5", conn)
df


,id_venta,fecha_operacion,cantidad_unidades,precio_venta_unitario,costo_unitario,descuento_pct,stock_inicial,stock_final,merma_unidades,canal,...,tiendas_tipo_local,productos_nombre,productos_categoria,productos_proveedor,productos_costo_unitario,productos_precio_lista,clientes_nombre,clientes_distrito,clientes_segmento,clientes_fecha_registro
0,1,2026-01-22,67,15.06,10.04,8,82,15,0,Mostrador,...,Tienda de mercado,Cafe instantaneo,Bebidas,Cafe Sierra,9.70,14.87,Marco Quispe,Los Olivos,Vecino frecuente,27/12/2025
1,2,2026-01-04,37,9.42,5.94,8 aprox,223,186,0,WhatsApp,...,Tienda de mercado,Papel higienico,Limpieza,Suave Hogar,5.76,9.30,Ana Soto,La Molina,Vecino frecuente,2025-12-11
2,3,2026-02-13,8,5.44,3.46,0,98,87,3 aprox,Mostrador,...,ALMACEN,Lenteja 500g,Menestras,Campo Bueno,3.38,5.14,Marco Mendoza,Ate,Ocasional,2025-08-23
3,4,06/06/2026,44,12.76,8.24,3,167,123,0,Telefono,...,Almacen,Aceite vegetal 1L,Aceites,Oleo Andino,8.03,12.76,Miguel Mendoza,Callao,Familia,2025-10-24
4,5,2026-02-19,61,9.28,5.96,0,180,118,1,Mostrador,...,Almacen,Papel higienico,Limpieza,Suave Hogar,5.76,9.30,Camila Castillo,San Miguel,Negocio,2025-12-18


**Pregunta 1. ¿Por qué en la Semana 6 se trabaja desde `ventas_original` y no desde tablas normalizadas?**

> **Respuesta:** Porque el objetivo del laboratorio dirigido 6 es implementar un pipeline automatizado que de por resultado datos limpios y reutilizables, siendo éste además, la implementación básica de gobierno de datos en cuanto a versionado y trazabilidad.

---
## Actividad 2. Carga completa de `ventas_original`

In [ ]:
ventas_original = pd.read_sql_query("SELECT * FROM ventas_original;", conn)
ventas_original.head()


,id_venta,fecha_operacion,cantidad_unidades,precio_venta_unitario,costo_unitario,descuento_pct,stock_inicial,stock_final,merma_unidades,canal,...,tiendas_tipo_local,productos_nombre,productos_categoria,productos_proveedor,productos_costo_unitario,productos_precio_lista,clientes_nombre,clientes_distrito,clientes_segmento,clientes_fecha_registro
0,1,2026-01-22,67,15.06,10.04,8,82,15,0,Mostrador,...,Tienda de mercado,Cafe instantaneo,Bebidas,Cafe Sierra,9.70,14.87,Marco Quispe,Los Olivos,Vecino frecuente,27/12/2025
1,2,2026-01-04,37,9.42,5.94,8 aprox,223,186,0,WhatsApp,...,Tienda de mercado,Papel higienico,Limpieza,Suave Hogar,5.76,9.30,Ana Soto,La Molina,Vecino frecuente,2025-12-11
2,3,2026-02-13,8,5.44,3.46,0,98,87,3 aprox,Mostrador,...,ALMACEN,Lenteja 500g,Menestras,Campo Bueno,3.38,5.14,Marco Mendoza,Ate,Ocasional,2025-08-23
3,4,06/06/2026,44,12.76,8.24,3,167,123,0,Telefono,...,Almacen,Aceite vegetal 1L,Aceites,Oleo Andino,8.03,12.76,Miguel Mendoza,Callao,Familia,2025-10-24
4,5,2026-02-19,61,9.28,5.96,0,180,118,1,Mostrador,...,Almacen,Papel higienico,Limpieza,Suave Hogar,5.76,9.30,Camila Castillo,San Miguel,Negocio,2025-12-18


**Pregunta 2. ¿Qué columnas presentan mayor riesgo de calidad y por qué?**

> **Respuesta:** A partir de la simple observación de la tabla de ventas original, se puede decir que las columnas precio_venta_unitario y costo_unitario presentan un mayor riesgo debido a la introducción parcial del símbolo de moneda, las diferencias en el separador de cifras decimales y a que los valores de ambas columnas se usan para hacer cálculos. Otras columnas que presentan riesgos de calidad son las de descuento, merma y las de fechas.

---
## Actividad 3. Diagnóstico inicial de calidad

In [ ]:
print("Filas y columnas:", ventas_original.shape)
display(ventas_original.info())

diagnostico = pd.DataFrame({
    "tipo_dato": ventas_original.dtypes.astype(str),
    "nulos": ventas_original.isna().sum(),
    "porcentaje_nulos": (ventas_original.isna().mean() * 100).round(2),
    "unicos": ventas_original.nunique(dropna=True)
})
diagnostico.sort_values("porcentaje_nulos", ascending=False)


Filas y columnas: (500, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_venta                  500 non-null    int64  
 1   fecha_operacion           486 non-null    object 
 2   cantidad_unidades         485 non-null    object 
 3   precio_venta_unitario     500 non-null    object 
 4   costo_unitario            500 non-null    object 
 5   descuento_pct             486 non-null    object 
 6   stock_inicial             489 non-null    object 
 7   stock_final               483 non-null    object 
 8   merma_unidades            491 non-null    object 
 9   canal                     485 non-null    object 
 10  metodo_pago               477 non-null    object 
 11  monto_venta_soles         499 non-null    object 
 12  margen_venta_soles        500 non-null    object 
 13  observacion               459 non-nul

None

,tipo_dato,nulos,porcentaje_nulos,unicos
observacion,object,41,8.2,21
metodo_pago,object,23,4.6,25
stock_final,object,17,3.4,228
clientes_nombre,object,17,3.4,78
clientes_segmento,object,16,3.2,13
canal,object,15,3.0,20
cantidad_unidades,object,15,3.0,79
descuento_pct,object,14,2.8,19
fecha_operacion,object,14,2.8,211
stock_inicial,object,11,2.2,243


**Pregunta 3. ¿Qué señales muestran que la tabla está desnormalizada?**

> **Respuesta:** De acuerdo a la función info() y la tabla de diagnóstico, las señales que indican que se trata de una tabla no normalizada son las 24 columnas, de un total de 27, que tienen asignado un tipo incorrecto de dato; y la significativa cantidad de valores nulos en 13 de las 27 columnas.

---
## Actividad 4. Revisión de duplicados y consistencia básica

In [ ]:
duplicados = ventas_original.duplicated().sum()
print("Duplicados exactos:", duplicados)

columnas_texto = ventas_original.select_dtypes(include="object").columns
resumen_texto = ventas_original[columnas_texto].nunique().sort_values(ascending=False)
resumen_texto


Duplicados exactos: 0


,0
monto_venta_soles,479
margen_venta_soles,474
precio_venta_unitario,363
costo_unitario,288
stock_inicial,243
stock_final,228
fecha_operacion,211
clientes_fecha_registro,84
cantidad_unidades,79
clientes_nombre,78


**Pregunta 4. ¿Qué problemas pueden aparecer si no se corrigen formatos de texto, fecha y número?**

> **Respuesta:** Los formatos de texto y fecha no corregidos pueden ocasionar registros repetidos que un algoritmo interprete como operaciones diferentes cuando en realidad no lo son; y los formatos de número no corregidos pueden dar resultados incorrectos o incluso imposibilitar todo cálculo numérico.

---
## Actividad 5. Funciones de limpieza

In [ ]:
def limpiar_texto(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

def convertir_numero(serie):
    return pd.to_numeric(serie, errors="coerce")

def convertir_fecha_mixta(serie):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        fecha_1 = pd.to_datetime(serie, errors="coerce", dayfirst=False)
        fecha_2 = pd.to_datetime(serie, errors="coerce", dayfirst=True)
    return fecha_1.fillna(fecha_2)


**Pregunta 5. Justifique una decisión de imputación aplicada en el laboratorio.**

> **Respuesta:** Las tres acciones de imputación de la actividad 5 están justificadas por los problemas que genera el dejar los datos como están, no obstante, puede decirse que la decisión de cambiar el tipo de dato de los atributos numéricos tiene un impacto ligeramante mayor al posibilitar los cálculos numéricos necesarios para analizar el estado de la tienda.

---
## Actividad 6. Corrección de tipos y formatos

In [ ]:
df_limpio = ventas_original.copy()

for col in df_limpio.select_dtypes(include="object").columns:
    if "fecha" in col.lower():
        df_limpio[col] = convertir_fecha_mixta(df_limpio[col])
    else:
        df_limpio[col] = limpiar_texto(df_limpio[col])

columnas_numericas = [
    "cantidad_unidades", "precio_venta_unitario", "costo_unitario",
    "descuento_pct", "stock_inicial", "stock_final", "merma_unidades",
    "monto_venta_soles", "margen_venta_soles", "productos_costo_unitario",
    "productos_precio_lista"
]

for col in columnas_numericas:
    if col in df_limpio.columns:
        df_limpio[col] = convertir_numero(df_limpio[col])

df_limpio.dtypes


,0
id_venta,int64
fecha_operacion,datetime64[ns]
cantidad_unidades,Int64
precio_venta_unitario,Float64
costo_unitario,Float64
descuento_pct,Int64
stock_inicial,Int64
stock_final,Int64
merma_unidades,Int64
canal,string[python]


**Pregunta 6. ¿Qué diferencia hay entre eliminar duplicados y corregir inconsistencias de contenido?**

> **Respuesta:** Corregir errores de contenido elimina caracteres no deseados en los valores de las celdas y uniformiza su formato y tipo de contenido; eliminar o descartar duplicados remueve un registro entero o un conjunto de ellos en el dataset analizado.

---
## Actividad 7. Imputación de valores faltantes

In [ ]:
nulos_antes = df_limpio.isna().sum()

for col in df_limpio.select_dtypes(include=["number"]).columns:
    mediana = df_limpio[col].median()
    df_limpio[col] = df_limpio[col].fillna(mediana)

for col in df_limpio.select_dtypes(include=["string", "object"]).columns:
    moda = df_limpio[col].mode(dropna=True)
    valor = moda.iloc[0] if len(moda) > 0 else "Sin Dato"
    df_limpio[col] = df_limpio[col].fillna(valor)

for col in df_limpio.select_dtypes(include=["datetime64[ns]"]).columns:
    mediana_fecha = df_limpio[col].dropna().median()
    if pd.isna(mediana_fecha):
        mediana_fecha = pd.Timestamp("1900-01-01")
    df_limpio[col] = df_limpio[col].fillna(mediana_fecha)

nulos_despues = df_limpio.isna().sum()

pd.DataFrame({
    "nulos_antes": nulos_antes,
    "nulos_despues": nulos_despues,
    "nulos_corregidos": nulos_antes - nulos_despues
}).query("nulos_antes > 0 or nulos_despues > 0")


,nulos_antes,nulos_despues,nulos_corregidos
fecha_operacion,81,0,81
cantidad_unidades,15,0,15
precio_venta_unitario,2,0,2
costo_unitario,1,0,1
descuento_pct,33,0,33
stock_inicial,23,0,23
stock_final,39,0,39
merma_unidades,23,0,23
canal,15,0,15
metodo_pago,23,0,23


**Pregunta 7. ¿Por qué se usa IQR para detectar outliers y cuándo no conviene eliminar registros?**

> **Respuesta:** El rango intercuartil se usa para detectar valores atípicos porque combina cálculos precisos para agrupar datos con el uso del diagrama de cajas, que visualmente señala los datos que se encuentran fuera de los bigotes, definidos por los límites superior e inferior, como atípicos. No conviene eliminar registros, en primer lugar, cuando éstos no están duplicados; tampoco es conveniente cuando no se analiza la acción o la posible razón de que se presenten nulos en un registro(patrón de ausencia), teniendo en cuenta que la eliminación de registros por una celda con valor nulo puede llevar a una pérdida de hasta un 60% de los datos.

---
## Actividad 8. Tratamiento de duplicados

In [ ]:
filas_antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
filas_despues = len(df_limpio)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Duplicados eliminados:", filas_antes - filas_despues)


Filas antes: 500
Filas después: 500
Duplicados eliminados: 0


**Pregunta 8. ¿Para qué sirve normalizar variables numéricas en una etapa posterior de análisis?**

> **Respuesta:** Sirve para comparar apropiadamente variables numéricas que presentan una escala muy diferente entre sí y de esta manera no se presente un sesgo que distorsione los resultados.

---
## Actividad 9. Detección y tratamiento de outliers con IQR

In [ ]:
columnas_outliers = [
    col for col in [
        "cantidad_unidades", "precio_venta_unitario", "costo_unitario",
        "descuento_pct", "stock_inicial", "stock_final", "merma_unidades",
        "monto_venta_soles", "margen_venta_soles", "productos_costo_unitario",
        "productos_precio_lista"
    ]
    if col in df_limpio.columns
]

reporte_outliers = []

for col in columnas_outliers:
    q1 = df_limpio[col].quantile(0.25)
    q3 = df_limpio[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    mascara = (df_limpio[col] < lim_inf) | (df_limpio[col] > lim_sup)
    reporte_outliers.append([col, int(mascara.sum()), lim_inf, lim_sup])
    df_limpio[col] = df_limpio[col].astype(float).clip(lower=lim_inf, upper=lim_sup)

pd.DataFrame(reporte_outliers, columns=["columna", "outliers_detectados", "limite_inferior", "limite_superior"])


,columna,outliers_detectados,limite_inferior,limite_superior
0,cantidad_unidades,6,-31.37500,113.62500
1,precio_venta_unitario,45,-6.31875,23.83125
2,costo_unitario,45,-3.75500,15.00500
3,descuento_pct,0,-15.00000,25.00000
4,stock_inicial,1,-61.00000,389.00000
5,stock_final,0,-78.00000,322.00000
6,merma_unidades,118,0.00000,0.00000
7,monto_venta_soles,36,-415.39875,1077.45125
8,margen_venta_soles,31,-160.35875,407.75125
9,productos_costo_unitario,45,-4.07000,15.29000


**Pregunta 9. Interprete el reporte antes/después: ¿la calidad del dataset mejoró? Sustente.**

> **Respuesta:** Dado que la penúltima línea del bloque de código anterior toma los valores atípicos y los iguala según su proximidad al límite inferior o superior, se puede decir que la calidad del dataset mejoró notablemente, porque ahora se tiene una distribución normal, sin sesgos. Se añade un bloque de código adicional, asistido por IA, para la comprobación visual.

In [ ]:
reporte_post_recorte = []

for fila in reporte_outliers:
    col = fila[0]
    lim_inf = fila[2]
    lim_sup = fila[3]

    nueva_mascara = (df_limpio[col] < lim_inf) | (df_limpio[col] > lim_sup)

    reporte_post_recorte.append([col, int(nueva_mascara.sum()), lim_inf, lim_sup])

df_post_recorte = pd.DataFrame(reporte_post_recorte, columns=["columna", "outliers_restantes", "limite_inferior", "limite_superior"])
df_post_recorte


,columna,outliers_restantes,limite_inferior,limite_superior
0,cantidad_unidades,0,-31.37500,113.62500
1,precio_venta_unitario,0,-6.31875,23.83125
2,costo_unitario,0,-3.75500,15.00500
3,descuento_pct,0,-15.00000,25.00000
4,stock_inicial,0,-61.00000,389.00000
5,stock_final,0,-78.00000,322.00000
6,merma_unidades,0,0.00000,0.00000
7,monto_venta_soles,0,-415.39875,1077.45125
8,margen_venta_soles,0,-160.35875,407.75125
9,productos_costo_unitario,0,-4.07000,15.29000


---
## Actividad 10. Normalización Min-Max

In [ ]:
df_normalizado = df_limpio.copy()

for col in columnas_outliers:
    minimo = df_normalizado[col].min()
    maximo = df_normalizado[col].max()
    if maximo != minimo:
        df_normalizado[col + "_minmax"] = (df_normalizado[col] - minimo) / (maximo - minimo)
    else:
        df_normalizado[col + "_minmax"] = 0

df_normalizado[[col for col in df_normalizado.columns if col.endswith("_minmax")]].describe().round(3)


,cantidad_unidades_minmax,precio_venta_unitario_minmax,costo_unitario_minmax,descuento_pct_minmax,stock_inicial_minmax,stock_final_minmax,merma_unidades_minmax,monto_venta_soles_minmax,margen_venta_soles_minmax,productos_costo_unitario_minmax,productos_precio_lista_minmax
count,500.000,500.000,500.000,500.000,500.000,500.000,500.0,500.000,500.000,500.000,500.000
mean,0.341,0.305,0.314,0.391,0.366,0.486,0.0,0.340,0.330,0.304,0.287
std,0.206,0.279,0.280,0.338,0.195,0.267,0.0,0.286,0.283,0.278,0.278
min,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000
25%,0.173,0.109,0.126,0.000,0.203,0.270,0.0,0.125,0.118,0.114,0.091
50%,0.347,0.213,0.223,0.333,0.368,0.489,0.0,0.255,0.234,0.212,0.197
75%,0.504,0.465,0.476,0.667,0.522,0.692,0.0,0.475,0.471,0.468,0.455
max,1.000,1.000,1.000,1.000,1.000,1.000,0.0,1.000,1.000,1.000,1.000


**Pregunta 10. ¿Qué tabla queda creada en SQLite y cómo se podría usar en el siguiente laboratorio?**

> **Respuesta:** La tabla que queda creada y guardada en SQLite es "ventas_original_limpia_s6". Para el siguiente laboratorio, contribuye de manera significativa al aportar datos completamente limpios y sobre todo normalizados, útiles para la creación de nuevas variables a partir de la ya existentes, y para alimentar a modelos de Machine Learning, incluso a redes neuronales.

---
## Actividad 11. Reporte antes/después

In [ ]:
reporte_final = pd.DataFrame({
    "metrica": ["filas", "columnas", "nulos_totales", "duplicados_exactos"],
    "antes": [
        ventas_original.shape[0],
        ventas_original.shape[1],
        int(ventas_original.isna().sum().sum()),
        int(ventas_original.duplicated().sum())
    ],
    "despues": [
        df_normalizado.shape[0],
        df_normalizado.shape[1],
        int(df_normalizado.isna().sum().sum()),
        int(df_normalizado.duplicated().sum())
    ]
})

reporte_final


,metrica,antes,despues
0,filas,500,500
1,columnas,27,38
2,nulos_totales,203,0
3,duplicados_exactos,0,0


**Pregunta 11. ¿Cómo demostrarías que la limpieza mejoró la calidad del dataset? ¿Qué métricas son las más relevantes?**

> **Respuesta:** Demostraría que la limpieza mejoró la calidad del dataset valiéndome de las dos últimas filas de reporte_final, que son a mi parecer, las métricas más importantes, y que indican que ahora ya no tengo nulos ni duplicados en mi dataset. Además, también podría usar el bloque de código adicional de la actividad 9 para demostrar que el dataset ya tampoco tiene valores atípicos.

---
## Actividad 12. Exportación del dataset limpio

In [ ]:
df_normalizado.to_sql("ventas_original_limpia_s6", conn, if_exists="replace", index=False)
df_normalizado.to_csv("ventas_original_limpia_s6.csv", index=False, encoding="utf-8-sig")

validacion = pd.read_sql_query("SELECT COUNT(*) AS filas_limpias FROM ventas_original_limpia_s6;", conn)
validacion


,filas_limpias
0,500


**Pregunta 12. ¿Por qué es importante guardar el dataset limpio como tabla nueva en lugar de reemplazar la original?**

> **Respuesta:** Es importante como buena práctica y parte de la documentación necesaria para auditar y sustentar todo el proceso realizado.

---
## Conclusiones

**Conclusión 1:** Se logró hacer un diagnóstico de calidad sobre una tabla de ventas no normalizada, detectando numerosos problemas que afectaban la confiabilidad de un análisis.

>

**Conclusión 2:** A través de la corrección de formatos, imputación de valores faltantes, tratamiento de duplicados y outliers(método IQR), con una ulterior normalización(Min-Max), se logró obtener un dataset limpio y adecuado para su uso y análisis.

>

**Conclusión 3:** Se demuestra que la limpieza de datos es un proceso muy importante, tanto para generar métricas y reportes directos, como para su implementación en modelos predictivos de Machine Learning, con un impacto directo en las decisiones de negocio. El archivo ventas_original_limpia_s6 asegura que el pipeline pueda reproducirse.